# 04 · Limiar, morfologia e o índice cardiotorácico

Aula **"Limiar e morfologia"**: máscara, Otsu, erosão e dilatação, abertura e fechamento, contornos e
área ("contar feijão"), e **"quando a régua chega"**: luz estável, fundo simples, medir sem rede.

O médico diagnostica cardiomegalia no raio-X de frente pelo **índice cardiotorácico (ICT)**: a largura
máxima do coração dividida pela largura interna máxima do tórax. **Acima de 0,5**, coração aumentado.
É uma razão entre larguras, então não depende da escala da imagem.

1. **tentativa 1, a receita das aulas**: Otsu → abertura → componentes → fechamento → pulmões → ICT;
2. **por que falha**: costelas e fundo;
3. **tentativa 2, o perfil por linha**: um limiar à meia altura em cada linha;
4. **escolher os parâmetros só na validação**;
5. **o resultado no teste**: o ICT separa as classes? E o corte clínico de 0,5?
6. **onde erra**.

Limitação desde já: o ICT de 0,5 vale para raio-X **PA** (de pé, de costas para o filme). No **AP**
(deitado, no leito) o coração sai ampliado. A base não informa a incidência.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import itertools

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve

from src import config
from src.data.dados import baixar_base, carregar_imagens, dividir, listar_imagens
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import ict

configurar_estilo()
SECAO = "04-limiar-morfologia"

baixar_base()
catalogo = dividir(listar_imagens())
imagens = carregar_imagens(catalogo["caminho"])
classes = catalogo["classe"].to_numpy()
conjunto = catalogo["conjunto"].to_numpy()

def mostrar(ax, imagem, titulo, **kw):
    ax.imshow(imagem, cmap=kw.pop("cmap", "gray"), **kw)
    ax.set_title(titulo, fontsize=9)
    ax.axis("off")

EXEMPLO = int(np.where(catalogo["id"] == "train/true/1298")[0][0])

## 1. Tentativa 1: a receita das aulas

No raio-X o **ar é escuro** e osso, coração e partes moles são claros. Os pulmões deveriam ser os dois
"feijões" escuros. Passo a passo, como nos exercícios de Otsu e morfologia:

In [ ]:
img = imagens[EXEMPLO]
suave = cv2.GaussianBlur(img, (5, 5), 0)
_, fixo = cv2.threshold(suave, 127, 255, cv2.THRESH_BINARY_INV)
limiar_otsu, otsu = cv2.threshold(suave, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
aberta = cv2.morphologyEx(otsu, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
n, rotulos = cv2.connectedComponents(aberta)
pulmoes = ict.segmentar_pulmoes(img)
ict_mascara = ict.ict_pela_mascara(pulmoes)

fig, eixos = plt.subplots(1, 6, figsize=(17, 3.3))
mostrar(eixos[0], img, "Raio-X")
mostrar(eixos[1], fixo, "Limiar fixo 127")
mostrar(eixos[2], otsu, f"Otsu (limiar {limiar_otsu:.0f})")
mostrar(eixos[3], aberta, "Abertura 3×3")
mostrar(eixos[4], rotulos, f"{n - 1} componentes", cmap="nipy_spectral")
mostrar(eixos[5], pulmoes, f"2 maiores + fechamento\nICT = {ict_mascara:.2f}")
salvar_figura(fig, SECAO, "receita-passo-a-passo")
plt.show()

In [ ]:
validacao = conjunto == "validacao"
ict_2d = np.array([ict.ict_pela_mascara(ict.segmentar_pulmoes(im)) for im in imagens[validacao]])
y_val = classes[validacao]
ok = np.isfinite(ict_2d)
tentativa1 = pd.Series({
    "falhas (sem 2 pulmões)": 1 - ok.mean(),
    "AUC": roc_auc_score(y_val[ok], ict_2d[ok]),
    "mediana ICT normal": np.median(ict_2d[ok & (y_val == 0)]),
    "mediana ICT cardiomegalia": np.median(ict_2d[ok & (y_val == 1)]),
}, name="segmentação 2D (validação)")
salvar_tabela(tentativa1.to_frame(), SECAO, "tentativa1")
tentativa1.round(3)

## 2. Por que falha

Doze raio-X de validação com a máscara de Otsu e a máscara final. O pulmão **não é um feijão sobre
mesa lisa**: as **costelas**, claras, cortam o campo pulmonar em faixas; vasos e coração escondem a
parte de baixo; e onde o pulmão encosta na axila escura, ele **vaza para o fundo** e é descartado
junto. Sobra pulmão aos pedaços, o tórax medido fica estreito e o ICT incha.

In [ ]:
rng = np.random.default_rng(config.SEMENTE)
amostra = rng.choice(np.where(validacao)[0], 6, replace=False)
fig, eixos = plt.subplots(3, 6, figsize=(15, 8))
for col, i in enumerate(amostra):
    _, m_otsu = cv2.threshold(cv2.GaussianBlur(imagens[i], (5, 5), 0), 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    m_final = ict.segmentar_pulmoes(imagens[i])
    valor = ict.ict_pela_mascara(m_final)
    mostrar(eixos[0, col], imagens[i], f"{catalogo['id'][i]}\n{config.ROTULOS[classes[i]]}")
    mostrar(eixos[1, col], m_otsu, "Otsu")
    mostrar(eixos[2, col], m_final, f"pulmões · ICT {valor:.2f}" if np.isfinite(valor) else "falhou")
salvar_figura(fig, SECAO, "receita-falhas")
plt.show()

## 3. Tentativa 2: o perfil por linha

O mesmo raciocínio do perfil de Sobel do notebook 02, agora para medir:

* um **desfoque forte só na vertical** (σ ≈ 3–6 px em y, 1,5 px em x) mistura cada costela com o espaço
  entre costelas e as apaga, mas preserva as bordas verticais do coração e da parede;
* em cada linha da metade de baixo do tórax, o perfil horizontal tem um **platô claro no meio**
  (coração), dois **vales** (pulmões) e duas **subidas** nas laterais (parede torácica);
* cada borda fica onde a intensidade cruza a **meia altura** entre platô e vale: um limiar por linha,
  que se adapta à exposição de cada exame;
* uma linha só vale se a medida é **fisicamente possível**: cada campo pulmonar ocupa ao menos 10 % do
  tórax (fora ficam as linhas do diafragma, onde os pulmões já acabaram) e o coração ao menos 15 %
  (fora ficam linhas em que o "platô" é só a coluna ou um fio);
* coração e tórax de cada exame são um quantil das larguras por linha, robusto a linhas ruins.

> **Nota de método.** As duas regras de validade entraram depois que a galeria de erros de uma
> primeira rodada, **no teste**, mostrou medidas impossíveis (ICT 0,00 e linhas do diafragma).
> A escolha dos parâmetros foi refeita só na validação (AUC de validação 0,704 → 0,723), mas o teste
> deixou de ser totalmente virgem para essas duas regras. Elas são critérios físicos, não ajustes
> finos, e o efeito foi medido na validação.

In [ ]:
LINHA = 70
borrada = cv2.GaussianBlur(img.astype(np.float32), (0, 0), sigmaX=1.5, sigmaY=4)
bordas_linha = ict.medir_linha(borrada[LINHA], 64)
medida = ict.medir_ict(img)

fig = plt.figure(figsize=(15, 7))
g = fig.add_gridspec(2, 3, height_ratios=[1.3, 1])
mostrar(fig.add_subplot(g[0, 0]), img, "Raio-X")
mostrar(fig.add_subplot(g[0, 1]), borrada, "Desfoque vertical: costelas somem")
ax = fig.add_subplot(g[0, 2])
mostrar(ax, img, f"Bordas por linha · ICT {medida.ict:.2f}")
for r, pe, ce, cd, pd_ in medida.bordas:
    ax.plot([ce, cd], [r, r], color=config.CORAL, lw=0.8, alpha=0.7)
    ax.plot([pe, pd_], [r + 0.4, r + 0.4], color=config.AZUL_CLARO, lw=0.4, alpha=0.6)
ax.axhline(LINHA, color=config.AMARELO, lw=1)
ax = fig.add_subplot(g[1, :])
ax.plot(img[LINHA].astype(int), color="#AAB4BE", lw=1, label="linha original")
ax.plot(borrada[LINHA], color=config.AZUL_PROFUNDO, lw=2, label="linha desfocada")
for x, nome, cor in zip(bordas_linha, ["parede", "coração", "coração", "parede"],
                        [config.AZUL_CLARO, config.CORAL, config.CORAL, config.AZUL_CLARO]):
    ax.axvline(x, color=cor, ls="--")
    ax.text(x, 252, nome, ha="center", fontsize=8, color=cor)
ax.set(xlabel="coluna (pixel)", ylabel="intensidade", ylim=(0, 265),
       title=f"Linha {LINHA}: coração {bordas_linha[2] - bordas_linha[1]} px · tórax {bordas_linha[3] - bordas_linha[0]} px")
ax.legend(loc="lower center")
salvar_figura(fig, SECAO, "perfil-por-linha",
              pd.DataFrame(medida.bordas, columns=["linha", "parede_esq", "coracao_esq", "coracao_dir", "parede_dir"]))
plt.show()

## 4. Escolher os parâmetros só na validação

Três escolhas: o desfoque vertical, a faixa de linhas medidas e o quantil das larguras. A grade é
avaliada **só na validação** (o teste fica guardado). Regra: entre as configurações a até 0,01 da
melhor AUC, fica a que **falha menos** (menos exames sem medida).

In [ ]:
grade = []
for sigma, linhas, quantil in itertools.product([3, 4, 6], [(50, 90), (55, 95), (60, 100)], [50, 75]):
    v = np.array([ict.medir_ict(im, sigma, linhas, quantil).ict for im in imagens[validacao]])
    ok = np.isfinite(v)
    grade.append({"sigma_vertical": sigma, "linhas": f"{linhas[0]}–{linhas[1]}", "quantil": quantil,
                  "auc": roc_auc_score(y_val[ok], v[ok]), "falhas": 1 - ok.mean(),
                  "_linhas": linhas})
grade = pd.DataFrame(grade).sort_values("auc", ascending=False)
perto = grade[grade["auc"] >= grade["auc"].max() - 0.01]
escolha = perto.sort_values(["falhas", "auc"], ascending=[True, False]).iloc[0]
PARAMS = dict(sigma_vertical=float(escolha["sigma_vertical"]), linhas=escolha["_linhas"],
              quantil=float(escolha["quantil"]))
print("Escolha:", PARAMS)
salvar_tabela(grade.drop(columns="_linhas").reset_index(drop=True), SECAO, "grade-validacao")
grade.drop(columns="_linhas").head(8).round(3)

## 5. O resultado no teste

Com os parâmetros fixados, o ICT é medido em **toda a base** (fica gravado em
`data/processed/ict.csv` para comparar com a CNN depois). O limiar de decisão tem duas versões:
o **clínico, 0,5**, e o que **maximiza sensibilidade + especificidade na validação** (Youden).
O teste só é olhado agora. O intervalo da AUC vem de 2.000 reamostragens bootstrap do teste.

In [ ]:
medidas = [ict.medir_ict(im, **PARAMS) for im in imagens]
catalogo["ict"] = [m.ict for m in medidas]
catalogo["coracao_px"] = [m.coracao_px for m in medidas]
catalogo["torax_px"] = [m.torax_px for m in medidas]
catalogo[["id", "conjunto", "classe", "ict", "coracao_px", "torax_px"]].to_csv(
    config.DATA_PROCESSED / "ict.csv", sep=";", index=False)

def avaliar(v, y, limiar):
    ok = np.isfinite(v)
    previsto = v[ok] > limiar
    real = y[ok] == 1
    return {"limiar": limiar, "falhas": 1 - ok.mean(), "acuracia": (previsto == real).mean(),
            "sensibilidade": previsto[real].mean(), "especificidade": (~previsto[~real]).mean()}

v_val = catalogo.loc[validacao, "ict"].to_numpy()
ok = np.isfinite(v_val)
fpr, tpr, cortes = roc_curve(y_val[ok], v_val[ok])
LIMIAR_YOUDEN = float(cortes[np.argmax(tpr - fpr)])

teste = conjunto == "teste"
v_te, y_te = catalogo.loc[teste, "ict"].to_numpy(), classes[teste]
ok_te = np.isfinite(v_te)
auc_te = roc_auc_score(y_te[ok_te], v_te[ok_te])
rng = np.random.default_rng(config.SEMENTE)
idx = np.where(ok_te)[0]
boot = []
for _ in range(2000):
    s = rng.choice(idx, len(idx))
    if len(np.unique(y_te[s])) == 2:
        boot.append(roc_auc_score(y_te[s], v_te[s]))
ic = np.percentile(boot, [2.5, 97.5])
print(f"AUC no teste: {auc_te:.3f} (IC 95 % {ic[0]:.3f}–{ic[1]:.3f}) | limiar de Youden na validação: {LIMIAR_YOUDEN:.3f}")

resultado = pd.DataFrame([
    {"regra": "clínica (ICT > 0,5)", **avaliar(v_te, y_te, 0.5)},
    {"regra": "Youden na validação", **avaliar(v_te, y_te, LIMIAR_YOUDEN)},
]).set_index("regra")
resultado["auc"] = auc_te
salvar_tabela(resultado, SECAO, "resultado-teste")
resultado.round(3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
bins = np.linspace(0.2, 0.8, 41)
for k in (0, 1):
    ax1.hist(v_te[ok_te & (y_te == k)], bins=bins, alpha=0.6, color=config.CORES_CLASSE[k],
             label=config.ROTULOS[k])
ax1.axvline(0.5, color="#555", ls="--", label="0,5 (clínico)")
ax1.axvline(LIMIAR_YOUDEN, color=config.ROXO, ls=":", label=f"{LIMIAR_YOUDEN:.2f} (Youden)")
ax1.set(xlabel="ICT medido", ylabel="exames de teste", title="ICT por classe (teste)")
ax1.legend()
fpr_te, tpr_te, _ = roc_curve(y_te[ok_te], v_te[ok_te])
ax2.plot(fpr_te, tpr_te, color=config.CORAL, lw=2, label=f"ICT clássico · AUC {auc_te:.3f}")
ax2.plot([0, 1], [0, 1], color="#999", ls="--", lw=1)
ax2.set(xlabel="1 − especificidade", ylabel="sensibilidade", title="Curva ROC (teste)")
ax2.legend(loc="lower right")
salvar_figura(fig, SECAO, "ict-teste", pd.DataFrame({"ict": v_te, "classe": y_te}))
plt.show()

## 6. Onde erra

Quatro grupos do teste: acertos de cada classe e os erros mais extremos (normais com o maior ICT e
cardiomegalias com o menor). Em vermelho as bordas do coração em cada linha medida; em azul, as
paredes.

In [ ]:
def desenhar(ax, i, titulo):
    m = medidas[i]
    mostrar(ax, imagens[i], titulo)
    for r, pe, ce, cd, pd_ in m.bordas:
        ax.plot([ce, cd], [r, r], color=config.CORAL, lw=0.7, alpha=0.6)
        ax.plot([pe, pe + 1], [r, r], color=config.AZUL_CLARO, lw=1.5)
        ax.plot([pd_ - 1, pd_], [r, r], color=config.AZUL_CLARO, lw=1.5)

idx_te = np.where(teste & np.isfinite(catalogo["ict"].to_numpy()))[0]
v = catalogo["ict"].to_numpy()
grupos = {
    "Normal, ICT baixo (acerto)": idx_te[np.argsort(v[idx_te] + 10 * classes[idx_te])][:4],
    "Cardiomegalia, ICT alto (acerto)": idx_te[np.argsort(-v[idx_te] - 10 * classes[idx_te])][:4],
    "Normal com ICT alto (erro)": idx_te[np.argsort(-v[idx_te] + 10 * classes[idx_te])][:4],
    "Cardiomegalia com ICT baixo (erro)": idx_te[np.argsort(v[idx_te] - 10 * classes[idx_te])][:4],
}
fig, eixos = plt.subplots(4, 4, figsize=(12, 13))
for linha, (nome, ids) in enumerate(grupos.items()):
    for col, i in enumerate(ids):
        desenhar(eixos[linha, col], i, f"{catalogo['id'][i]} · ICT {v[i]:.2f}")
    eixos[linha, 0].text(-8, 64, nome, rotation=90, ha="right", va="center", fontsize=10,
                         fontweight="bold", color=config.AZUL_PROFUNDO)
salvar_figura(fig, SECAO, "onde-erra")
plt.show()

## Conclusões

* **A receita "contar feijão" não serve para pulmão:** Otsu + abertura + componentes + fechamento
  deixou 38 % dos exames de validação sem dois pulmões e, nos outros, ICT inflado (mediana 0,67 nos
  normais, 0,69 na cardiomegalia) e sem separar as classes (AUC 0,55). As costelas cortam o pulmão,
  e o pulmão vaza para o fundo pela axila.
* **O perfil por linha mede:** desfoque vertical para apagar costelas e limiar à meia altura em cada
  linha. No teste, a mediana do ICT é **0,43 nos normais e 0,51 na cardiomegalia**, praticamente em
  cima do corte clínico de 0,5, e só 0,7 % dos exames fica sem medida.
* **Separa as classes, mas não muito:** AUC **0,739** no teste (IC 95 % 0,710–0,769). Com o corte
  clínico de 0,5: acurácia 67,8 %, sensibilidade **54 %**, especificidade **82 %**. Com o limiar de
  Youden escolhido na validação (0,476): acurácia 69,3 %, sensibilidade 69 %, especificidade 70 %.
* **Onde erra:** linhas do diafragma que ainda passam pelas regras, enquadramentos diferentes (a faixa
  de linhas é fixa em pixels, e um tórax pequeno no meio do filme fica fora dela), pacientes rodados,
  e o que nenhuma régua resolve: a incidência AP, que amplia o coração, e o **ruído de rótulo** visto
  no notebook 01.
* **O valor de uma régua clássica:** cada número é explicável. Dá para apontar na imagem onde está a
  borda do coração e por que o ICT deu 0,62. É contra esse 0,739 explicável que a CNN do notebook 05
  vai ser comparada: o ICT de cada exame ficou em `data/processed/ict.csv`.